In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from stargazer.stargazer import Stargazer, LineLocation

from img_labormarket.config import BLD, SRC
from img_labormarket.analysis.help_analysis import *
from img_labormarket.data_management.help_data import *
# Setze die Anzeigeoptionen für Pandas
pd.set_option('display.float_format', '{:.3f}'.format)

green = '#3498db'
red = '#d35400'

In [ ]:
df = pd.read_csv(BLD / 'data' / 'clean_data.csv')
# Quick Fix
df['first_wage'] = df['real_first_wage']
df['log_real_first_wage'] = df['log_first_wage']

1. Degree FE + Gender
2. Gender * Img
3. Industry FE
4. University
5. Study Field?
6. Graduation Year


In [ ]:
df = df.loc[~df['Gender'].isin(['Prefer not to say'])]
df = restrict_wage(df,'real_first_wage', lower=10, upper=150)
#df = df.loc[df['first_wage']>=df['min_wage']]
res0 = smf.ols("log_real_first_wage ~ img + C(Gender, Treatment('Male')) + C(DegreeCompact)", data=df).fit(cov_type='HC1')

res1 = smf.ols("log_real_first_wage ~ img + C(Gender, Treatment('Male')) + C(DegreeCompact) + C(new_age)", data=df).fit(cov_type='HC1')

res2 = smf.ols("log_real_first_wage ~ img + C(Gender, Treatment('Male')) + C(DegreeCompact) + C(StudyFieldCluster) + C(new_age)", data=df).fit(cov_type='HC1')

res3 = smf.ols("log_real_first_wage ~ img + C(Gender, Treatment('Male')) + C(DegreeCompact)+ C(StudyFieldCluster) + C(StartJobYear) + C(new_age)", data=df).fit(cov_type='HC1')

res4 = smf.ols("log_real_first_wage ~ img*C(Gender, Treatment('Male')) + C(DegreeCompact)+ + C(new_age)", data=df).fit(cov_type='HC1')

res5 = smf.ols("log_real_first_wage ~ img + C(Gender, Treatment('Male')) + C(StudyFieldCluster) + C(sector_coarse) + C(DegreeCompact) + + C(new_age)", data=df).fit(cov_type='HC1')

res6 = smf.ols("log_real_first_wage ~ img + C(Gender, Treatment('Male')) + C(StudyFieldCluster) + C(sector_coarse) + C(DegreeCompact) + C(StartJobYear) + + C(new_age)", data=df).fit(cov_type='HC1')
# Prepare the models
models = [res1, res2, res3, res5, res6]

# Define restriction labels
extras = [
    "Year FE",
    "Industry FE",
]

# Define whether restrictions are applied (x: yes, blank: no)
fixed_effects = [
    #["", ""],  # res0: 
    ["", ""],  # res1: 
    ["", ""],  # res2:
    #["", ""],  # res4: 
    ["x", ""],  # res3: 
    ["", "x"],  # res5: 
    ["x", "x"]  # res6: 
]

# Create Stargazer table
stargazer = Stargazer(models)
#stargazer.title("OLS Regression Results with Restrictions")
#stargazer.custom_columns(["Model 0", "Model 1", "Model 2", "Model 3", "Model 4"], [1, 1, 1, 1, 1])

# Add restrictions as additional rows
for idx, restriction in enumerate(extras):
    row_values = [fixed_effects[model_idx][idx] for model_idx in range(len(models))]
    stargazer.add_line(restriction, row_values)

stargazer.rename_covariates({
    "img":"Immigrant",
    "Intercept": "Constant",
    "C(DegreeCompact)[T.Master]": "Master",
    "C(DegreeCompact)[T.PhD]": "PhD",
    "C(StudyFieldCluster)[T.Engineering & CS]": "Engineering \& CS",
    "C(StudyFieldCluster)[T.Natural & Life Sciences]": "Natural \& Life Sciences",
    "C(Gender, Treatment('Male'))[T.Female]": "Female",
    "img:C(Gender, Treatment('Male'))[T.Female]": "Immigrant x Female"   
})
stargazer.covariate_order(["img",
                           "C(Gender, Treatment('Male'))[T.Female]",
                           "C(DegreeCompact)[T.Master]",
                           "C(DegreeCompact)[T.PhD]",
                           "C(StudyFieldCluster)[T.Engineering & CS]",
                           "C(StudyFieldCluster)[T.Natural & Life Sciences]",
                           #'GPA',
                           #"img:C(Gender, Treatment('Male'))[T.Female]",
                           ])
# Output LaTeX table
stargazer.show_residual_std_err = False
stargazer.show_f_statistic = False
stargazer.dependent_variable_name = "Log Wage"
code = stargazer.render_latex()
with open(BLD / 'tables' / 'mincer_table.tex', 'w') as f:
    f.write(code)
stargazer


In [ ]:

plt.hist(df.loc[df['img']==1,'first_wage'], bins=20,color='orange' ,alpha=0.5, density=True, label='Immigrants')
plt.axvline(df.loc[df['img']==1,'first_wage'].mean(), color='orange', linestyle='dashed', linewidth=1)
plt.hist(df.loc[df['img']==0,'first_wage'], bins=20,color='blue', alpha=0.5, density=True, label='Native')
plt.axvline(df.loc[df['img']==0,'first_wage'].mean(), color='blue', linestyle='dashed', linewidth=1)
plt.legend()
plt.show()

In [ ]:
# Create figure and subplots
fig, ax = plt.subplots(nrows=3, ncols=1, figsize=(20, 9), sharex=True, sharey=True)

# Define study fields
dfm = df.loc[df['DegreeCompact']=='Master']
study_fields = {
    "Natural & Life Sciences": dfm.loc[dfm['StudyFieldCluster'] == 'Natural & Life Sciences'],
    "Engineering & CS": dfm.loc[dfm['StudyFieldCluster'] == 'Engineering & CS'],
    "Business & Economics": dfm.loc[dfm['StudyFieldCluster'] == 'Business & Economics']
}

colors = {"immigrant": red, "native": green}

# Iterate over subplots
for i, (title, data) in enumerate(study_fields.items()):
    if data.empty:
        continue  # Skip empty study fields to avoid errors
    show = f'{title} (N = {len(data)})'
    ax[i].set_title(show, fontsize=26)  # Thicker title
    ax[i].set_ylabel("Density", fontsize=22)  # Thicker y-axis label

    for group, label, color in [(1, "Immigrants", colors["immigrant"]), (0, "Native", colors["native"])]:
        wage_data = data.loc[data['img'] == group, 'first_wage']
        
        if wage_data.empty:
            continue  # Skip if there are no values for this group
        
        # Compute histogram and density normalization
        no_bins = 10 if title == "Natural & Life Sciences" else 10
        counts, bins = np.histogram(wage_data.dropna(), bins=no_bins)  # Drop NaN values
        if counts.sum() == 0:
            continue  # Skip plotting if no valid data

        weights = counts / (counts.sum() * np.diff(bins))  # Normalize to density

        # Plot histogram with thicker edges
        ax[i].hist(bins[:-1], bins=bins, color=color, weights=weights, alpha=0.5, label=label, linewidth=2.5)
        
        # Mean wage line (thicker)
        ax[i].axvline(wage_data.mean(), color=color, linestyle='dashed', linewidth=4,label=f'Mean Wage')

    ax[i].tick_params(axis='both', which='major', labelsize=20, width=1.5)  # Thicker ticks


ax[0].legend(fontsize=20)  # Increase legend font size
# X-axis label with larger, bolder font
plt.xlabel("Wage (in 1000€)", fontsize=22)
#plt.set_ylabel("Frequency", fontsize=18)  # Y-axis label with larger, bolder font

#plt.savefig(BLD / 'figures' / 'wage_distribution_by_study_field.pdf', bbox_inches='tight')
plt.show()

# Mincer Regression

In [ ]:
x = df.groupby(['StudyFieldCluster','img'])[['StudyField','sector_coarse']].value_counts()

In [ ]:
df = df.loc[~df['Gender'].isin(['Prefer not to say','Divers'])]
res = smf.ols("log_real_first_wage ~ img + C(Gender, Treatment('Male')) + C(DegreeCompact)", data=df).fit(cov_type='HC1')
res.summary()

8% Gap but not close to any standard significance level.

In [ ]:
df = df.loc[~df['Gender'].isin(['Prefer not to say','Divers'])]
res1 = smf.ols("log_first_wage ~ img + C(Gender, Treatment('Male')) + C(DegreeCompact)", data=df).fit(cov_type='HC1')

res2 = smf.ols("log_first_wage ~ img + C(Gender, Treatment('Male')) + C(DegreeCompact) + C(StudyFieldCluster)", data=df).fit(cov_type='HC1')

res3 = smf.ols("log_first_wage ~ img + C(Gender, Treatment('Male')) + C(DegreeCompact) + C(GradYear)", data=df).fit(cov_type='HC1')

res4 = smf.ols("log_first_wage ~ img + C(Gender, Treatment('Male')) + C(DegreeCompact)*GPA*C(StudyFieldCluster)", data=df).fit(cov_type='HC1')

res5 = smf.ols("log_first_wage ~ img + C(Gender, Treatment('Male')) + C(StudyFieldCluster) + C(sector_coarse) + C(DegreeCompact)", data=df).fit(cov_type='HC1')

res6 = smf.ols("log_first_wage ~ img + C(Gender, Treatment('Male')) + C(StudyFieldCluster) + C(sector_coarse) + C(DegreeCompact) + C(GradYear)", data=df).fit(cov_type='HC1')
# Prepare the models
models = [res1, res2, res3, res4, res5, res6]

# Define restriction labels
extras = [
    "Graduation Year FE",
    "Industry FE",
]

# Define whether restrictions are applied (x: yes, blank: no)
fixed_effects = [
    ["", ""],  # res1: 
    ["", ""],  # res2:
    ["x", ""],  # res3: 
    ["", ""],  # res4: 
    ["", "x"]  # res5: 
    ["x", "x"]  # res6: 
]

# Create Stargazer table
stargazer = Stargazer(models)
stargazer.title("OLS Regression Results with Restrictions")
#stargazer.custom_columns(["Model 0", "Model 1", "Model 2", "Model 3", "Model 4"], [1, 1, 1, 1, 1])

# Add restrictions as additional rows
for idx, restriction in enumerate(restrictions):
    row_values = [applied_restrictions[model_idx][idx] for model_idx in range(len(models))]
    stargazer.add_line(restriction, row_values)


# Output LaTeX table
stargazer


### Restrict Wage

In [ ]:
#Cut below 15 and above 150
df = restrict_wage(df,'first_wage', lower=0, upper=150)
df = df.loc[df['first_wage']>df['min_wage']]
res = smf.ols("log_first_wage ~ img + C(Gender, Treatment('Male')) + C(DegreeCompact)", data=df).fit(cov_type='HC1')
res.summary()

Gap gets smaller. Gap at about 5% about half of the women gap.

In [ ]:
#Cut below 15 and above 150
df = restrict_wage(df,'first_wage', lower=15, upper=150)
res = smf.ols("log_first_wage ~ img*C(Gender, Treatment('Male')) + C(DegreeCompact)", data=df).fit(cov_type='HC1')
res.summary()

Some evidence for intersectionality between gender and immigrant status.

In [ ]:
res = smf.ols("log_first_wage ~ img + C(Gender, Treatment('Male')) + C(DegreeCompact) + C(GradYear)", data=df).fit(cov_type='HC1')
res.summary()

In [ ]:
res = smf.ols("log_first_wage ~ img + C(Gender, Treatment('Male')) + C(DegreeCompact) + C(GradYear) + C(StudyFieldCluster)", data=df).fit(cov_type='HC1')
res.summary()

In [ ]:
res = smf.ols("log_first_wage ~ img + C(Gender, Treatment('Male')) + C(DegreeCompact)*GPA+C(StudyFieldCluster)", data=df).fit(cov_type='HC1')
res.summary()

### Restrict to Master students

In [ ]:
dfm = df.loc[df['DegreeCompact']=='Master']
res = smf.ols("log_first_wage ~ img + C(Gender, Treatment('Male')) + C(StudyFieldCluster)", data=dfm).fit(cov_type='HC1')
res.summary()

Gap stays stable but p-value of course look worse.

In [ ]:
dfm.loc[dfm['StudyFieldCluster'] == 'Natural & Life Sciences'].groupby('img')['sector'].value_counts()

In [ ]:
x = dfm.loc[dfm['StudyFieldCluster'] == 'Natural & Life Sciences'][['img','first_wage','sector','StudyField']]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create figure and subplots
fig, ax = plt.subplots(nrows=3, ncols=1, figsize=(20, 15), sharex=True, sharey=True)

# Define study fields
study_fields = {
    "Natural & Life Sciences": dfm.loc[dfm['StudyFieldCluster'] == 'Natural & Life Sciences'],
    "Engineering & CS": dfm.loc[dfm['StudyFieldCluster'] == 'Engineering & CS'],
    "Business & Economics": dfm.loc[dfm['StudyFieldCluster'] == 'Business & Economics']
}

colors = {"immigrant": "orange", "native": "blue"}

# Iterate over subplots
for i, (title, data) in enumerate(study_fields.items()):
    if data.empty:
        continue  # Skip empty study fields to avoid errors
    
    ax[i].set_title(title, fontsize=16)  # Thicker title

    for group, label, color in [(1, "Immigrants", colors["immigrant"]), (0, "Native", colors["native"])]:
        wage_data = data.loc[data['img'] == group, 'first_wage']
        
        if wage_data.empty:
            continue  # Skip if there are no values for this group
        
        # Compute histogram and density normalization
        no_bins = 10 if title == "Natural & Life Sciences" else 20
        counts, bins = np.histogram(wage_data.dropna(), bins=no_bins)  # Drop NaN values
        if counts.sum() == 0:
            continue  # Skip plotting if no valid data

        weights = counts / (counts.sum() * np.diff(bins))  # Normalize to density

        # Plot histogram with thicker edges
        ax[i].hist(bins[:-1], bins=bins, color=color, weights=weights, alpha=0.5, label=label, linewidth=1.5)
        
        # Mean wage line (thicker)
        ax[i].axvline(wage_data.mean(), color=color, linestyle='dashed', linewidth=2)

    ax[i].legend(fontsize=14)  # Increase legend font size
    ax[i].tick_params(axis='both', which='major', labelsize=14, width=1.5)  # Thicker ticks

# X-axis label with larger, bolder font
plt.xlabel("Wage (in 1000€)", fontsize=16)
plt.ylabel("Frequency", fontsize=16)  # Y-axis label with larger, bolder font
plt.show()
#plt.savefig(BLD / 'figures' / 'wage_distribution_by_study_field.pdf')

### Industry Effects

In [ ]:
rslt = dfm.groupby('sector')[['first_wage','Q44r1']].mean()
rslt['pct_increase'] = (rslt['Q44r1']-rslt['first_wage'])/rslt['first_wage']*100
rslt

In [ ]:
rslt = dfm.groupby('new_sector')[['first_wage','Q44r1']].mean()
rslt['pct_increase'] = (rslt['Q44r1']-rslt['first_wage'])/rslt['first_wage']*100
rslt

Many of the other fields can be grouped in here too. 

In [ ]:
df['sector'].value_counts()

In [ ]:
dfm = df.loc[df['DegreeCompact']=='Master']
res = smf.ols("log_first_wage ~ img + C(Gender, Treatment('Male')) + C(StudyFieldCluster) + C(sector_coarse) + C(DegreeCompact)", data=df).fit(cov_type='HC1')
res.summary()

In [ ]:
df['sector_coarse'].value_counts()

In [ ]:
dfm.groupby('img')['sector'].value_counts().unstack(0)

In [ ]:
dfm = df.loc[df['DegreeCompact']=='Master']
res = smf.ols("log_first_wage ~ img + C(Gender, Treatment('Male')) + C(StudyFieldCluster) + C(new_sector) + C(DegreeCompact)", data=df).fit(cov_type='HC1')
res.summary()

If we control for sector FEs the gap gets larger (for women it gets smaller), it seems to suggest that immigrants target high-wage sectors but within industry gaps are sizeable.

# Characteristics of the first job

### Sector or Industry

In [ ]:
df = dfm
df.groupby('img')['new_sector'].value_counts(normalize=True).unstack(0)

### Using English at first job?

In [ ]:
df = add_codings(df, mapping_df, 'Q32r2')
df.groupby(['img'])['Q32r2_label'].value_counts(normalize=True).unstack(0)

### Salary met my expectations

In [ ]:
df.groupby('img')['Q24r3'].value_counts(normalize=True).unstack()

### How many employees does company of your first job have?

1	Less than 20 employees

2	Between 20 to 50 employees

3	Between 50 to 200 employees

4	Between 200 to 500 employees

5	Between 500 to 1,000 employees

6	Between 1,000 to 5,000 employees

7	Between 5,000 to 10,000 employees

8	More than 10,000

In [ ]:
df.groupby('img')['Q33'].value_counts(normalize=True).unstack()

### Requirement of your first job?

1	No requirement

2	High School Degree

3	Vocational Training

4	Bachelor Degree

5	Master Degree or higher

6	I don’t know

In [ ]:
df.groupby('img')['Q39'].value_counts(normalize=True).unstack(0)

In [ ]:
df.groupby('img')['job_requirement_college_degree'].value_counts(normalize=True).unstack(0)

In [ ]:
df.groupby(['img','job_requirement_college_degree'])['first_wage'].mean().unstack(0)